In [18]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the path to the dataset
dataset_path = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single'

# Check if the path exists and list the first few files
if os.path.exists(dataset_path):
    print(f"Successfully accessed: {dataset_path}")
    files = os.listdir(dataset_path)
    print(f"Total items found: {len(files)}")
    print("Sample files:", files[:5])
else:
    print(f"Path not found: {dataset_path}. Please ensure the folder name is correct.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully accessed: /content/drive/MyDrive/Colab Notebooks/MVSA_Single
Total items found: 7
Sample files: ['data', 'labelResultAll.txt', 'mvsa_single_processed.parquet', 'checkpoints', 'vilbert_final']


In [19]:

def saveParquet(df):
    """Fungsi untuk menyimpan DataFrame ke format Parquet."""
    # Tentukan path penyimpanan di Google Drive
    parquet_save_path = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed.parquet'
    try:
        df.to_parquet(parquet_save_path, index=False)
        print(f"Dataset berhasil disimpan ke: {parquet_save_path}")
        print(f"Ukuran file: {os.path.getsize(parquet_save_path) / (1024*1024):.2f} MB")
    except Exception as e:
        print(f"Terjadi kesalahan saat menyimpan: {e}")


In [20]:
import pandas as pd
import os
from PIL import Image
import io
import gc
from concurrent.futures import ThreadPoolExecutor

# Paths
labels_file = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single/labelResultAll.txt'
data_dir = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single/data'
# Path ke file yang baru saja disimpan
parquet_path = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed.parquet'

def process_single_row(row):
    """Helper function to process one image-text pair with resizing."""
    try:
        item_id = str(int(row['id']))
        image_path = os.path.join(data_dir, f"{item_id}.jpg")
        text_path = os.path.join(data_dir, f"{item_id}.txt")

        t_label = str(row['text']).strip()
        i_label = str(row['image']).strip()

        # Sentiment logic
        if t_label == "neutral":
            f_label = i_label
        elif i_label == 'neutral':
            f_label = t_label
        elif t_label == i_label:
            f_label = t_label
        else:
            return None

        if os.path.exists(image_path) and os.path.exists(text_path):
            # Process and Resize Image to 384x384 (standard for ViLT)
            with Image.open(image_path) as img:
                img = img.convert('RGB')
                img = img.resize((384, 384), Image.Resampling.LANCZOS)
                buf = io.BytesIO()
                img.save(buf, format='PNG')
                img_bytes = buf.getvalue()

            # Process Text
            with open(text_path, 'r', encoding='latin-1') as file:
                text_content = file.read().strip()

            return {
                'ID': item_id,
                'text_sentiment': t_label,
                'image_sentiment': i_label,
                'final_sentiment': f_label,
                'image_bytes': img_bytes,
                'text_content': text_content
            }
    except Exception as e:
        print(f"Error processing row: {e}")
        return None
    return None

# Cek apakah file parquet sudah ada
if os.path.exists(parquet_path):
    print(f"File parquet sudah ada, memuat dari: {parquet_path}")
    df = pd.read_parquet(parquet_path)
    print(f"Total data dimuat: {len(df)}")
    print("\nDistribusi sentimen:")
    print(df['final_sentiment'].value_counts())
    display(df.head())
else:
    print(f"File parquet tidak ditemukan, memulai pemrosesan...")
    print(f"Membaca labels dari: {labels_file}")

    # 1. Read labels
    df_labels = pd.read_csv(labels_file, sep=',', header=0)
    print(f"Total labels dibaca: {len(df_labels)}")
    print(f"Nama kolom dalam file labels: {list(df_labels.columns)}")
    print(f"Sample data:\n{df_labels.head()}")

    # 2. Parallel processing
    print("\nMemulai pemrosesan paralel dengan resizing gambar ke 384x384...")
    rows = [row for _, row in df_labels.iterrows()]

    with ThreadPoolExecutor(max_workers=4) as executor:
        results = list(executor.map(process_single_row, rows))

    # 3. Filter results and cleanup
    processed_records = [r for r in results if r is not None]
    print(f"Total records berhasil diproses: {len(processed_records)}")

    if len(processed_records) > 0:
        df = pd.DataFrame(processed_records)

        # 4. Simpan ke parquet
        print(f"Menyimpan data ke: {parquet_path}")
        df.to_parquet(parquet_path, index=False)
        print("Data berhasil disimpan!")

        del results, processed_records, df_labels, rows
        gc.collect()

        print(f"\nTotal data berhasil diproses dan disimpan: {len(df)}")
        print("\nDistribusi sentimen:")
        print(df['final_sentiment'].value_counts())
        display(df.head())
        saveParquet(df)
    else:
        print("PERINGATAN: Tidak ada data yang berhasil diproses!")
        print("Periksa apakah:")
        print(f"  1. File label ada di: {labels_file}")
        print(f"  2. Folder data ada di: {data_dir}")
        print(f"  3. File .jpg dan .txt ada di dalam folder data")

File parquet sudah ada, memuat dari: /content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed.parquet
Total data dimuat: 4511

Distribusi sentimen:
final_sentiment
positive    2683
negative    1358
neutral      470
Name: count, dtype: int64


,ID,text_sentiment,image_sentiment,final_sentiment,image_bytes,text_content
0,1,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,How I feel today #legday #jelly #aching #gym
1,2,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,grattis min griskulting!!!???? va bara tvungen...
2,3,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,RT @polynminion: The moment I found my favouri...
3,4,positive,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,#escort We have a young and energetic team and...
4,5,positive,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,"RT @chrisashaffer: Went to SSC today to be a ""..."


In [21]:
# ─────────────────────────────────────────────
# Cell 1 — Install & Import Dependencies
# ─────────────────────────────────────────────
!pip install torch torchvision scikit-learn pyarrow tqdm gdown -q

import os, io, gc, re, random, warnings
import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import torchvision.models as tv_models
import torchvision.transforms as transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print("All libraries imported successfully.")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


All libraries imported successfully.
PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
GPU Memory      : 15.6 GB


In [22]:
# ─────────────────────────────────────────────
# Cell 2 — Model Configuration & Seed
# ─────────────────────────────────────────────
CONFIG = {
    # ── Paths ──────────────────────────────────────────────────────
    "parquet_path"       : "/content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed.parquet",
    "checkpoint_dir"     : "/content/drive/MyDrive/Colab Notebooks/MVSA_Single/lstm_checkpoints",
    "glove_path"         : "/content/drive/MyDrive/Colab Notebooks/glove.6B.300d.txt",

    # ── Text Encoder (GloVe + BiLSTM) ──────────────────────────────
    "embed_dim"          : 300,         # GloVe 6B 300d
    "glove_finetune"     : True,        # fine-tune embeddings during training
    "hidden_dim"         : 256,         # BiLSTM hidden units per direction → output 512
    "num_lstm_layers"    : 2,
    "max_seq_len"        : 50,

    # ── Image Encoder (ResNet50 + BiLSTM) ──────────────────────────
    "image_backbone"     : "resnet50",
    "image_feature_dim"  : 2048,        # ResNet50 last conv channels
    "image_proj_dim"     : 512,         # project 2048 → 512 to match d_model
    "image_input_size"   : 224,
    "resnet_finetune"    : True,        # fine-tune all ResNet layers

    # ── d_model (shared dim for co-attention) ──────────────────────
    "d_model"            : 512,         # BiLSTM output dim = hidden_dim × 2

    # ── Co-Attention ───────────────────────────────────────────────
    "co_attn_type"       : "multi-head",       # bidirectional cross-attention
    "num_heads"          : 8,           # MHA heads (512 / 8 = 64 per head)
    "co_attn_layers"     : 2,
    "co_attn_ffn_dim"    : 2048,        # FFN inner dim (4 × d_model)
    "co_attn_dropout"    : 0.1,

    # ── Fusion & Classifier ────────────────────────────────────────
    "fusion_strategy"    : "concatenation",    # cat(mean_text, mean_image)
    "fusion_input_dim"   : 512 * 2,     # 1024
    "classifier_hidden"  : 256,
    "num_classes"        : 3,           # negative=0 / neutral=1 / positive=2
    "classifier_dropout" : 0.3,

    # ── Training ────────────────────────────────────────────────────
    "epochs"             : 30,
    "batch_size"         : 32,
    "lr_main"            : 1e-4,        # LSTM, CoAttn, classifier
    "lr_resnet_finetune" : 1e-5,        # ResNet50 backbone
    "weight_decay"       : 1e-4,
    "gradient_clip"      : 1.0,
    "early_stop_patience": 5,
    "scheduler"          : "CosineAnnealingLR",

    # ── Data Split ──────────────────────────────────────────────────
    "train_ratio"        : 0.80,
    "val_ratio"          : 0.10,
    "test_ratio"         : 0.10,

    # ── Reproducibility ────────────────────────────────────────────
    "seed"               : 42,
    "num_workers"        : 2,
    "device"             : "cuda" if torch.cuda.is_available() else "cpu",
}

# ── Seed everything ──────────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)

# ── Pretty-print configuration ───────────────────────────────────────
SEP = "=" * 62
print(SEP)
print("   Multimodal LSTM + Co-Attention Sentiment — Configuration")
print(SEP)
print(f"  Device                  : {CONFIG['device'].upper()}")
if torch.cuda.is_available():
    print(f"  GPU                     : {torch.cuda.get_device_name(0)}")

print(f"\n  ── Text Encoder (GloVe + BiLSTM) ────────────────────")
print(f"  Embedding dim           : {CONFIG['embed_dim']}  (GloVe 6B)")
print(f"  GloVe fine-tune         : {CONFIG['glove_finetune']}")
print(f"  BiLSTM hidden per dir   : {CONFIG['hidden_dim']}  → output {CONFIG['d_model']}")
print(f"  BiLSTM layers           : {CONFIG['num_lstm_layers']}")
print(f"  Max sequence length     : {CONFIG['max_seq_len']}")

print(f"\n  ── Image Encoder (ResNet50 + BiLSTM) ────────────────")
print(f"  Backbone                : {CONFIG['image_backbone']} (pretrained ImageNet)")
print(f"  Feature dim (raw)       : {CONFIG['image_feature_dim']}")
print(f"  Projected dim           : {CONFIG['image_proj_dim']}")
print(f"  Input image size        : {CONFIG['image_input_size']}×{CONFIG['image_input_size']}")
print(f"  ResNet fine-tune        : {CONFIG['resnet_finetune']}")

print(f"\n  ── Co-Attention Mechanism ───────────────────────────")
print(f"  Type                    : {CONFIG['co_attn_type']} (bidirectional)")
print(f"  Number of heads         : {CONFIG['num_heads']}")
print(f"  Number of layers        : {CONFIG['co_attn_layers']}")
print(f"  d_model                 : {CONFIG['d_model']}")
print(f"  FFN hidden dim          : {CONFIG['co_attn_ffn_dim']}")
print(f"  Dropout                 : {CONFIG['co_attn_dropout']}")

print(f"\n  ── Fusion & Classifier ──────────────────────────────")
print(f"  Fusion strategy         : {CONFIG['fusion_strategy']}")
print(f"  Fusion input dim        : {CONFIG['fusion_input_dim']}")
print(f"  Classifier hidden       : {CONFIG['classifier_hidden']}")
print(f"  Num output classes      : {CONFIG['num_classes']}  (neg / neu / pos)")
print(f"  Classifier dropout      : {CONFIG['classifier_dropout']}")

print(f"\n  ── Training Hyperparameters ─────────────────────────")
print(f"  Epochs                  : {CONFIG['epochs']}")
print(f"  Batch size              : {CONFIG['batch_size']}")
print(f"  LR (main)               : {CONFIG['lr_main']}")
print(f"  LR (ResNet fine-tune)   : {CONFIG['lr_resnet_finetune']}")
print(f"  Weight decay            : {CONFIG['weight_decay']}")
print(f"  Gradient clip           : {CONFIG['gradient_clip']}")
print(f"  LR scheduler            : {CONFIG['scheduler']}")
print(f"  Early-stop patience     : {CONFIG['early_stop_patience']} epochs")

print(f"\n  ── Data ─────────────────────────────────────────────")
print(f"  Split (train/val/test)  : {CONFIG['train_ratio']}/{CONFIG['val_ratio']}/{CONFIG['test_ratio']}")
print(f"  Random seed             : {CONFIG['seed']}")
print(SEP)


   Multimodal LSTM + Co-Attention Sentiment — Configuration
  Device                  : CUDA
  GPU                     : Tesla T4

  ── Text Encoder (GloVe + BiLSTM) ────────────────────
  Embedding dim           : 300  (GloVe 6B)
  GloVe fine-tune         : True
  BiLSTM hidden per dir   : 256  → output 512
  BiLSTM layers           : 2
  Max sequence length     : 50

  ── Image Encoder (ResNet50 + BiLSTM) ────────────────
  Backbone                : resnet50 (pretrained ImageNet)
  Feature dim (raw)       : 2048
  Projected dim           : 512
  Input image size        : 224×224
  ResNet fine-tune        : True

  ── Co-Attention Mechanism ───────────────────────────
  Type                    : multi-head (bidirectional)
  Number of heads         : 8
  Number of layers        : 2
  d_model                 : 512
  FFN hidden dim          : 2048
  Dropout                 : 0.1

  ── Fusion & Classifier ──────────────────────────────
  Fusion strategy         : concatenation
  Fusion in

In [23]:
# ─────────────────────────────────────────────
# Cell 3 — Load Parquet, Build Vocab & GloVe Embeddings
# ─────────────────────────────────────────────

# ── 3a. Load parquet ─────────────────────────────────────────────────
print(f"Loading parquet from: {CONFIG['parquet_path']}")
df_full = pd.read_parquet(CONFIG["parquet_path"])
print(f"Total rows loaded : {len(df_full)}")
print(f"Columns           : {list(df_full.columns)}")

before = len(df_full)
df_full = df_full.dropna(subset=["text_content", "image_bytes", "final_sentiment"]).reset_index(drop=True)
print(f"Rows after dropna : {len(df_full)}  (dropped {before - len(df_full)})")

# ── 3b. Encode labels ────────────────────────────────────────────────
LABEL_MAP  = {"negative": 0, "neutral": 1, "positive": 2}
LABEL_IMAP = {v: k for k, v in LABEL_MAP.items()}

df_full["label"] = df_full["final_sentiment"].str.strip().map(LABEL_MAP)
if df_full["label"].isna().any():
    n = df_full["label"].isna().sum()
    print(f"WARN: {n} rows with unrecognised labels — dropping.")
    df_full = df_full.dropna(subset=["label"]).reset_index(drop=True)
df_full["label"] = df_full["label"].astype(int)

print(f"\nLabel distribution:")
for lbl, idx in LABEL_MAP.items():
    count = (df_full["label"] == idx).sum()
    print(f"  {idx}  {lbl:<10}  {count:>5}  ({count/len(df_full)*100:.1f}%)")

# ── 3c. Stratified split 80 / 10 / 10 ─────────────────────────────
df_train, df_tmp = train_test_split(
    df_full,
    test_size=(CONFIG["val_ratio"] + CONFIG["test_ratio"]),
    stratify=df_full["label"],
    random_state=CONFIG["seed"],
)
df_val, df_test = train_test_split(
    df_tmp,
    test_size=CONFIG["test_ratio"] / (CONFIG["val_ratio"] + CONFIG["test_ratio"]),
    stratify=df_tmp["label"],
    random_state=CONFIG["seed"],
)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f"\nData split (stratified):")
print(f"  Train : {len(df_train):>5} rows")
print(f"  Val   : {len(df_val):>5} rows")
print(f"  Test  : {len(df_test):>5} rows")

# ── 3d. Build vocabulary from TRAIN corpus only ──────────────────────
def simple_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
MIN_FREQ  = 2   # keep words appearing >=2 times

counter = Counter()
for text in df_train["text_content"]:
    counter.update(simple_tokenize(text))

vocab = [PAD_TOKEN, UNK_TOKEN] + [w for w, c in counter.items() if c >= MIN_FREQ]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
VOCAB_SIZE = len(vocab)
PAD_IDX    = word2idx[PAD_TOKEN]
UNK_IDX    = word2idx[UNK_TOKEN]
print(f"\nVocabulary size : {VOCAB_SIZE}  (min_freq={MIN_FREQ})")

# ── 3e. Load GloVe & build embedding matrix ──────────────────────────
import subprocess, zipfile

glove_path = CONFIG["glove_path"]
glove_dir  = os.path.dirname(glove_path)

# Auto-download from Stanford NLP if not present
if not os.path.exists(glove_path):
    glove_zip = os.path.join(glove_dir, "glove.6B.zip")
    os.makedirs(glove_dir, exist_ok=True)
    print("GloVe 6B not found — downloading from Stanford NLP (~822 MB)...")
    subprocess.run(
        ["wget", "-q", "--show-progress", "-O", glove_zip,
         "https://nlp.stanford.edu/data/glove.6B.zip"],
        check=True
    )
    print("Extracting glove.6B.300d.txt ...")
    with zipfile.ZipFile(glove_zip, "r") as z:
        z.extract("glove.6B.300d.txt", glove_dir)
    os.remove(glove_zip)
    print(f"GloVe saved to: {glove_path}")

print(f"Loading GloVe from: {glove_path}")
glove_vectors = {}
with open(glove_path, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Reading GloVe"):
        parts = line.rstrip().split(" ")
        glove_vectors[parts[0]] = np.array(parts[1:], dtype=np.float32)
print(f"GloVe vocab loaded: {len(glove_vectors):,} words")

embedding_matrix = np.zeros((VOCAB_SIZE, CONFIG["embed_dim"]), dtype=np.float32)
hits, misses = 0, 0
for word, idx in word2idx.items():
    if word in glove_vectors:
        embedding_matrix[idx] = glove_vectors[word]
        hits += 1
    else:
        embedding_matrix[idx] = np.random.normal(0, 0.01, CONFIG["embed_dim"])
        misses += 1

print(f"Embedding matrix : {embedding_matrix.shape}")
print(f"  GloVe hits     : {hits}  ({hits/VOCAB_SIZE*100:.1f}%)")
print(f"  Random init    : {misses}  ({misses/VOCAB_SIZE*100:.1f}%)")
del glove_vectors; gc.collect()


Loading parquet from: /content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed.parquet
Total rows loaded : 4511
Columns           : ['ID', 'text_sentiment', 'image_sentiment', 'final_sentiment', 'image_bytes', 'text_content']
Rows after dropna : 4511  (dropped 0)

Label distribution:
  0  negative     1358  (30.1%)
  1  neutral       470  (10.4%)
  2  positive     2683  (59.5%)

Data split (stratified):
  Train :  3608 rows
  Val   :   451 rows
  Test  :   452 rows

Vocabulary size : 4192  (min_freq=2)
Loading GloVe from: /content/drive/MyDrive/Colab Notebooks/glove.6B.300d.txt


Reading GloVe: 0it [00:00, ?it/s]

GloVe vocab loaded: 400,000 words
Embedding matrix : (4192, 300)
  GloVe hits     : 3733  (89.1%)
  Random init    : 459  (10.9%)


31

In [24]:
# ─────────────────────────────────────────────
# Cell 4 — Dataset Class & DataLoaders
# ─────────────────────────────────────────────

IMAGE_MEAN = [0.485, 0.456, 0.406]
IMAGE_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_input_size"], CONFIG["image_input_size"])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_input_size"], CONFIG["image_input_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD),
])


class MVSADataset(Dataset):
    """
    Multimodal dataset that reads from a parquet-derived DataFrame.
      image_bytes  : raw PNG bytes (already 384×384 in parquet → resized to 224 here)
      text_content : raw text string
      label        : integer 0/1/2
    """

    def __init__(self, dataframe, word2idx, max_len, image_transform, pad_idx, unk_idx):
        self.df        = dataframe.reset_index(drop=True)
        self.word2idx  = word2idx
        self.max_len   = max_len
        self.transform = image_transform
        self.pad_idx   = pad_idx
        self.unk_idx   = unk_idx

    def __len__(self):
        return len(self.df)

    def _tokenize_pad(self, text):
        tokens = simple_tokenize(str(text))[:self.max_len]
        ids    = [self.word2idx.get(t, self.unk_idx) for t in tokens]
        # pad / truncate to max_len
        ids    = ids + [self.pad_idx] * (self.max_len - len(ids))
        return torch.tensor(ids, dtype=torch.long), len(tokens)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ── Decode image from bytes ──────────────────────────
        raw = row["image_bytes"]
        raw = raw if isinstance(raw, bytes) else bytes(raw)
        img = Image.open(io.BytesIO(raw)).convert("RGB")
        img_tensor = self.transform(img)

        # ── Tokenise text ────────────────────────────────────
        token_ids, seq_len = self._tokenize_pad(row["text_content"])

        return {
            "image"   : img_tensor,                                      # [3, H, W]
            "text"    : token_ids,                                       # [max_len]
            "seq_len" : torch.tensor(seq_len, dtype=torch.long),
            "label"   : torch.tensor(row["label"], dtype=torch.long),
        }


# ── Instantiate datasets ─────────────────────────────────────────────
_ds_kwargs = dict(
    word2idx  = word2idx,
    max_len   = CONFIG["max_seq_len"],
    pad_idx   = PAD_IDX,
    unk_idx   = UNK_IDX,
)
train_dataset = MVSADataset(df_train, image_transform=train_transform, **_ds_kwargs)
val_dataset   = MVSADataset(df_val,   image_transform=eval_transform,  **_ds_kwargs)
test_dataset  = MVSADataset(df_test,  image_transform=eval_transform,  **_ds_kwargs)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"],
                          shuffle=True,  num_workers=CONFIG["num_workers"],
                          pin_memory=True, drop_last=False)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=CONFIG["num_workers"],
                          pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=CONFIG["num_workers"],
                          pin_memory=True)

print(f"Train batches : {len(train_loader):>4}  ({len(train_dataset)} samples)")
print(f"Val   batches : {len(val_loader):>4}  ({len(val_dataset)} samples)")
print(f"Test  batches : {len(test_loader):>4}  ({len(test_dataset)} samples)")

# ── Sanity-check one batch ───────────────────────────────────────────
_batch = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  image   : {_batch['image'].shape}   (expect [B, 3, 224, 224])")
print(f"  text    : {_batch['text'].shape}    (expect [B, {CONFIG['max_seq_len']}])")
print(f"  seq_len : {_batch['seq_len'].shape}  values sample={_batch['seq_len'][:4].tolist()}")
print(f"  label   : {_batch['label'].shape}   values sample={_batch['label'][:4].tolist()}")
del _batch


Train batches :  113  (3608 samples)
Val   batches :   15  (451 samples)
Test  batches :   15  (452 samples)

Batch shapes:
  image   : torch.Size([32, 3, 224, 224])   (expect [B, 3, 224, 224])
  text    : torch.Size([32, 50])    (expect [B, 50])
  seq_len : torch.Size([32])  values sample=[16, 10, 18, 23]
  label   : torch.Size([32])   values sample=[2, 2, 2, 2]


In [25]:

# ─────────────────────────────────────────────
# Cell 5 — Model Architecture
#   • TextEncoder     : GloVe Embed → BiLSTM
#   • ImageEncoder    : ResNet50 (no FC) → Linear proj → BiLSTM
#   • CoAttentionLayer: bidirectional 8-head MHA + FFN
#   • MultimodalSentimentModel: composing all above
# ─────────────────────────────────────────────

# ──────────────────────────────────────────────────────────────
# 5a. Text Encoder
# ──────────────────────────────────────────────────────────────
class TextEncoder(nn.Module):
    """
    GloVe 300d Embedding (fine-tunable) → 2-layer BiLSTM
    Output: [B, T, hidden*2]   (d_model = 512 by default)
    """

    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers,
                 pad_idx, pretrained_weights, finetune=True):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.embedding.weight = nn.Parameter(
            torch.from_numpy(pretrained_weights), requires_grad=finetune
        )
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.3 if num_layers > 1 else 0.0,
        )

    def forward(self, token_ids):
        """token_ids: [B, T]  →  [B, T, hidden*2]"""
        x = self.embedding(token_ids)         # [B, T, E]
        out, _ = self.lstm(x)                 # [B, T, H*2]
        return out


# ──────────────────────────────────────────────────────────────
# 5b. Image Encoder
# ──────────────────────────────────────────────────────────────
class ImageEncoder(nn.Module):
    """
    ResNet50 (pretrained, all layers unfrozen) → feature map [B, 2048, 7, 7]
    → flatten to [B, 49, 2048] → Linear 2048→proj_dim → 2-layer BiLSTM
    Output: [B, 49, hidden*2]  (d_model = 512 by default)
    """

    def __init__(self, proj_dim, hidden_dim, num_layers, finetune=True):
        super().__init__()
        resnet = tv_models.resnet50(weights=tv_models.ResNet50_Weights.DEFAULT)
        # Remove avgpool and fc; keep spatial feature maps from layer4
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])
        if not finetune:
            for p in self.backbone.parameters():
                p.requires_grad = False
        # Freeze BN running stats for stable training
        for m in self.backbone.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()

        self.proj = nn.Sequential(
            nn.Linear(2048, proj_dim),
            nn.LayerNorm(proj_dim),
        )
        self.lstm = nn.LSTM(
            input_size=proj_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.3 if num_layers > 1 else 0.0,
        )

    def forward(self, pixel_values):
        """pixel_values: [B, 3, H, W]  →  [B, 49, hidden*2]"""
        feat  = self.backbone(pixel_values)           # [B, 2048, 7, 7]
        flat  = feat.flatten(2).permute(0, 2, 1)      # [B, 49, 2048]
        proj  = self.proj(flat)                        # [B, 49, proj_dim]
        out, _ = self.lstm(proj)                       # [B, 49, hidden*2]
        return out

    def train(self, mode=True):
        super().train(mode)
        # Always keep BN in eval regardless of overall mode
        for m in self.backbone.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
        return self


# ──────────────────────────────────────────────────────────────
# 5c. Co-Attention Layer (bidirectional, multi-head)
# ──────────────────────────────────────────────────────────────
class CoAttentionLayer(nn.Module):
    """
    Bidirectional Cross-Modal Co-Attention:
      • Text stream : Q=text,  K/V=image  → attended text
      • Image stream: Q=image, K/V=text   → attended image
    Each stream: MHA → Add & LayerNorm → FFN(GELU) → Add & LayerNorm
    """

    def __init__(self, d_model, num_heads, ffn_dim, dropout):
        super().__init__()
        assert d_model % num_heads == 0, \
            f"d_model ({d_model}) must be divisible by num_heads ({num_heads})"

        # ── Text ← Image ──────────────────────────────────────
        self.t2i_attn  = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.t2i_norm1 = nn.LayerNorm(d_model)
        self.t2i_ffn   = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
        )
        self.t2i_norm2 = nn.LayerNorm(d_model)

        # ── Image ← Text ──────────────────────────────────────
        self.i2t_attn  = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.i2t_norm1 = nn.LayerNorm(d_model)
        self.i2t_ffn   = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
        )
        self.i2t_norm2 = nn.LayerNorm(d_model)

    def forward(self, text_feats, image_feats, text_pad_mask=None):
        """
        text_feats  : [B, T,  d_model]
        image_feats : [B, Lv, d_model]
        text_pad_mask : [B, T] bool — True = ignored padding position
        Returns: text_out [B, T, d], image_out [B, Lv, d]
        """
        # ── Guard: if any sample has ALL text positions masked (empty text),
        # unmasking that row prevents softmax(all -inf) = NaN in attention.
        if text_pad_mask is not None:
            all_masked = text_pad_mask.all(dim=1)          # [B]  True where row is all-PAD
            if all_masked.any():
                text_pad_mask = text_pad_mask.clone()
                text_pad_mask[all_masked] = False          # treat as unmasked (attend everywhere)

        # Text attends to Image
        t_out, _ = self.t2i_attn(text_feats, image_feats, image_feats)
        t_out     = self.t2i_norm1(text_feats + t_out)
        t_out     = self.t2i_norm2(t_out + self.t2i_ffn(t_out))

        # Image attends to Text
        i_out, _ = self.i2t_attn(image_feats, text_feats, text_feats,
                                  key_padding_mask=text_pad_mask)
        i_out     = self.i2t_norm1(image_feats + i_out)
        i_out     = self.i2t_norm2(i_out + self.i2t_ffn(i_out))

        return t_out, i_out


# ──────────────────────────────────────────────────────────────
# 5d. Full Multimodal Sentiment Model
# ──────────────────────────────────────────────────────────────
class MultimodalSentimentModel(nn.Module):
    """
    Full architecture:
      text  → TextEncoder  → [B, T,  512]  ─┐
                                              ├─ CoAttention ×2  ─→ mean-pool
      image → ImageEncoder → [B, 49, 512]  ─┘
                                              cat [B, 1024]
                                              LayerNorm → FC(1024→256) → ReLU
                                              → Dropout → FC(256→3)
    """

    def __init__(self, config, vocab_size, pad_idx, pretrained_weights):
        super().__init__()
        d       = config["d_model"]
        h       = config["hidden_dim"]
        nl      = config["num_lstm_layers"]

        self.text_encoder = TextEncoder(
            vocab_size=vocab_size,
            embed_dim=config["embed_dim"],
            hidden_dim=h,
            num_layers=nl,
            pad_idx=pad_idx,
            pretrained_weights=pretrained_weights,
            finetune=config["glove_finetune"],
        )
        self.image_encoder = ImageEncoder(
            proj_dim=config["image_proj_dim"],
            hidden_dim=h,
            num_layers=nl,
            finetune=config["resnet_finetune"],
        )
        self.co_attn_layers = nn.ModuleList([
            CoAttentionLayer(
                d_model=d,
                num_heads=config["num_heads"],
                ffn_dim=config["co_attn_ffn_dim"],
                dropout=config["co_attn_dropout"],
            )
            for _ in range(config["co_attn_layers"])
        ])

        fusion_in = config["fusion_input_dim"]   # 1024
        clf_hid   = config["classifier_hidden"]
        drop      = config["classifier_dropout"]

        self.classifier = nn.Sequential(
            nn.LayerNorm(fusion_in),
            nn.Linear(fusion_in, clf_hid),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(clf_hid, config["num_classes"]),
        )

    def forward(self, text_ids, images, text_pad_mask=None):
        """
        text_ids     : [B, T]
        images       : [B, 3, H, W]
        text_pad_mask: [B, T] bool (True = padding)
        Returns: logits [B, 3]
        """
        text_feats  = self.text_encoder(text_ids)       # [B, T, 512]
        image_feats = self.image_encoder(images)         # [B, 49, 512]

        for layer in self.co_attn_layers:
            text_feats, image_feats = layer(text_feats, image_feats,
                                            text_pad_mask=text_pad_mask)

        # Mean-pool both attended sequences
        if text_pad_mask is not None:
            # Masked mean for text (exclude padding)
            valid_mask  = (~text_pad_mask).float().unsqueeze(-1)   # [B, T, 1]
            text_pool   = (text_feats * valid_mask).sum(1) / valid_mask.sum(1).clamp(min=1)
        else:
            text_pool   = text_feats.mean(1)

        image_pool  = image_feats.mean(1)                          # [B, 512]
        fused       = torch.cat([text_pool, image_pool], dim=-1)   # [B, 1024]
        return self.classifier(fused)                               # [B, 3]

    def train(self, mode=True):
        super().train(mode)
        self.image_encoder.train(mode)   # keeps BN in eval
        return self


# ── Instantiate ──────────────────────────────────────────────────────
device = torch.device(CONFIG["device"])

model = MultimodalSentimentModel(
    config=CONFIG,
    vocab_size=VOCAB_SIZE,
    pad_idx=PAD_IDX,
    pretrained_weights=embedding_matrix,
).to(device)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_p:,}")
print(f"Trainable parameters: {trainable_p:,}")

# ── Quick forward-pass test ──────────────────────────────────────────
model.eval()
with torch.no_grad():
    _b = next(iter(train_loader))
    _txt = _b["text"].to(device)
    _img = _b["image"].to(device)
    _msk = (_txt == PAD_IDX)                       # [B, T] padding mask
    _logits = model(_txt, _img, text_pad_mask=_msk)
print(f"Forward pass OK  |  logits shape: {tuple(_logits.shape)}  (expect [B, 3])")
del _b, _txt, _img, _msk, _logits
model.train()


Total parameters    : 44,564,163
Trainable parameters: 44,564,163
Forward pass OK  |  logits shape: (32, 3)  (expect [B, 3])


MultimodalSentimentModel(
  (text_encoder): TextEncoder(
    (embedding): Embedding(4192, 300, padding_idx=0)
    (lstm): LSTM(300, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  )
  (image_encoder): ImageEncoder(
    (backbone): Sequential(
      (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (4): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3):

In [26]:

# ─────────────────────────────────────────────
# Cell 6 — Training Loop
# ─────────────────────────────────────────────
from sklearn.utils.class_weight import compute_class_weight as _cwf

# ── Optimizer: two LR groups ─────────────────────────────────────────
resnet_param_ids = set(id(p) for p in model.image_encoder.backbone.parameters())
resnet_params    = [p for p in model.parameters() if id(p) in resnet_param_ids]
main_params      = [p for p in model.parameters() if id(p) not in resnet_param_ids]

optimizer = AdamW([
    {"params": main_params,   "lr": CONFIG["lr_main"]},
    {"params": resnet_params, "lr": CONFIG["lr_resnet_finetune"]},
], weight_decay=CONFIG["weight_decay"])

scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"], eta_min=1e-6)

# ── Weighted loss to handle class imbalance ──────────────────────────
_cw = _cwf('balanced',
           classes=np.arange(CONFIG["num_classes"]),
           y=df_train["label"].values)
class_weights = torch.tensor(_cw, dtype=torch.float).to(device)
print(f"Class weights  ─  negative={_cw[0]:.3f}  neutral={_cw[1]:.3f}  positive={_cw[2]:.3f}")
criterion = nn.CrossEntropyLoss(weight=class_weights)

# ── Evaluation helper ────────────────────────────────────────────────
def evaluate(loader, model, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    nan_batches = 0
    with torch.no_grad():
        for batch in loader:
            text   = batch["text"].to(device)
            images = batch["image"].to(device)
            labels = batch["label"].to(device)
            mask   = (text == PAD_IDX)

            logits = model(text, images, text_pad_mask=mask)
            loss   = criterion(logits, labels)
            if torch.isnan(loss):
                nan_batches += 1
                continue
            total_loss += loss.item()
            all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    if nan_batches:
        print(f"  [WARN] {nan_batches} NaN-loss batches skipped in evaluation.")
    if len(all_labels) == 0:
        return float('nan'), 0.0, 0.0
    avg_loss = total_loss / (len(loader) - nan_batches)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, acc, f1


# ── Training loop ───────────────────────────────────────────────────
best_val_f1      = 0.0
patience_counter = 0
best_ckpt_path   = os.path.join(CONFIG["checkpoint_dir"], "lstm_best.pt")
history          = []
nan_batches_total = 0

header = (f"{'Epoch':>5}  {'Train Loss':>10}  {'Train Acc':>9}  "
          f"{'Val Loss':>8}  {'Val Acc':>7}  {'Val F1':>7}  {'LR(main)':>9}  {'Status'}")
print(header)
print("─" * len(header))

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()

    train_loss, train_preds, train_labels = 0.0, [], []
    nan_this_epoch = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:>2}/{CONFIG['epochs']}", leave=False)

    for batch in pbar:
        text   = batch["text"].to(device)
        images = batch["image"].to(device)
        labels = batch["label"].to(device)
        mask   = (text == PAD_IDX)

        optimizer.zero_grad()
        logits = model(text, images, text_pad_mask=mask)
        loss   = criterion(logits, labels)

        # Skip corrupted batches rather than letting NaN propagate
        if torch.isnan(loss):
            nan_this_epoch += 1
            nan_batches_total += 1
            pbar.set_postfix(loss="NaN-skip")
            continue

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CONFIG["gradient_clip"])
        optimizer.step()

        train_loss  += loss.item()
        train_preds.extend(logits.detach().argmax(dim=-1).cpu().tolist())
        train_labels.extend(labels.cpu().tolist())
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    scheduler.step()

    valid_batches = len(train_loader) - nan_this_epoch
    tr_loss = train_loss / valid_batches if valid_batches > 0 else float('nan')
    tr_acc  = accuracy_score(train_labels, train_preds) if train_labels else 0.0
    val_loss, val_acc, val_f1 = evaluate(val_loader, model, criterion, device)
    cur_lr  = optimizer.param_groups[0]["lr"]

    if nan_this_epoch:
        print(f"  [WARN] epoch {epoch}: {nan_this_epoch} NaN-loss batches skipped.")

    # Early stopping & checkpoint
    status = ""
    if val_f1 > best_val_f1:
        best_val_f1      = val_f1
        patience_counter = 0
        torch.save({
            "epoch"          : epoch,
            "model_state"    : model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_f1"         : val_f1,
            "val_acc"        : val_acc,
            "config"         : CONFIG,
        }, best_ckpt_path)
        status = "✓ best saved"
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["early_stop_patience"]:
            print(f"\nEarly stopping at epoch {epoch}  "
                  f"(no improvement for {CONFIG['early_stop_patience']} epochs)")
            break

    history.append({
        "epoch": epoch,
        "train_loss": tr_loss, "train_acc": tr_acc,
        "val_loss": val_loss,  "val_acc": val_acc,  "val_f1": val_f1,
    })

    print(f"{epoch:>5}  {tr_loss:>10.4f}  {tr_acc:>9.4f}  "
          f"{val_loss:>8.4f}  {val_acc:>7.4f}  {val_f1:>7.4f}  "
          f"{cur_lr:>9.2e}  {status}")

print(f"\nTraining complete.  Best val macro-F1 : {best_val_f1:.4f}")
if nan_batches_total:
    print(f"[INFO] Total NaN-loss batches skipped across all epochs: {nan_batches_total}")
print(f"Best checkpoint   : {best_ckpt_path}")


Class weights  ─  negative=1.107  neutral=3.199  positive=0.560
Epoch  Train Loss  Train Acc  Val Loss  Val Acc   Val F1   LR(main)  Status
───────────────────────────────────────────────────────────────────────────


Epoch  1/30:   0%|          | 0/113 [00:00<?, ?it/s]

    1      0.9568     0.5532    0.9914   0.4435   0.4248   9.97e-05  ✓ best saved


Epoch  2/30:   0%|          | 0/113 [00:00<?, ?it/s]

    2      0.7105     0.6924    0.9139   0.6674   0.5893   9.89e-05  ✓ best saved


...

Epoch  6/30:   0%|          | 0/113 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79218c2f1bc0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
        Exception ignored in:   <function _MultiProcessingDataLoaderIter.__del__ at 0x79218c2f1bc0> 
Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    ^^^if w.is_alive():^
^^ ^ ^^ ^^ ^^ ^ ^ ^^^^^^

    6      0.1826     0.9365    1.7357   0.6541   0.5529   9.05e-05  


Epoch  7/30:   0%|          | 0/113 [00:00<?, ?it/s]


Early stopping at epoch 7  (no improvement for 5 epochs)

Training complete.  Best val macro-F1 : 0.5893
Best checkpoint   : /content/drive/MyDrive/Colab Notebooks/MVSA_Single/lstm_checkpoints/lstm_best.pt


In [27]:
# ─────────────────────────────────────────────
# Cell 7 — Evaluation on Test Set
# ─────────────────────────────────────────────
import json

# ── Load best checkpoint ─────────────────────────────────────────────
print(f"Loading best checkpoint: {best_ckpt_path}")
ckpt = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
print(f"  Checkpoint epoch : {ckpt['epoch']}   "
      f"val_f1={ckpt['val_f1']:.4f}   val_acc={ckpt['val_acc']:.4f}")

# ── Predictions on test set ──────────────────────────────────────────
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        text   = batch["text"].to(device)
        images = batch["image"].to(device)
        labels = batch["label"].to(device)
        mask   = (text == PAD_IDX)

        logits = model(text, images, text_pad_mask=mask)
        probs  = F.softmax(logits, dim=-1)
        preds  = logits.argmax(dim=-1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        all_probs.extend(probs.cpu().tolist())

# ── Metrics ──────────────────────────────────────────────────────────
class_names = [LABEL_IMAP[i] for i in range(CONFIG["num_classes"])]
test_acc    = accuracy_score(all_labels, all_preds)
test_f1     = f1_score(all_labels, all_preds, average="macro", zero_division=0)

SEP = "=" * 62
print(f"\n{SEP}")
print("  Test Set Results — Multimodal LSTM + Co-Attention")
print(SEP)
print(f"  Accuracy  (overall)    : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  Macro F1  (unweighted) : {test_f1:.4f}")
print(f"\n  Per-Class Classification Report:")
print(classification_report(all_labels, all_preds,
                             target_names=class_names,
                             digits=4, zero_division=0))

# ── Confusion Matrix (text table) ────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
print("  Confusion Matrix  (rows=true, cols=pred):")
col_w = max(len(n) for n in class_names) + 2
header_row = " " * 12 + "  ".join(f"{n:>{col_w}}" for n in class_names)
print(header_row)
for i, row_vals in enumerate(cm):
    row_str = "  ".join(f"{v:{col_w}d}" for v in row_vals)
    print(f"  {class_names[i]:>10}  {row_str}")

# ── Training history summary ─────────────────────────────────────────
if history:
    hist_df = pd.DataFrame(history)
    print(f"\n  Training History (last 5 epochs):")
    print(hist_df.tail(5).to_string(index=False))

print(f"\n{SEP}")

# ── Save final model & artifacts ─────────────────────────────────────
SAVE_DIR = "/content/drive/MyDrive/Colab Notebooks/MVSA_Single/lstm_final"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save({
    "model_state_dict"    : model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config"              : CONFIG,
    "label_map"           : LABEL_MAP,
    "label_imap"          : LABEL_IMAP,
    "vocab"               : word2idx,
    "best_val_f1"         : best_val_f1,
    "test_acc"            : test_acc,
    "test_f1"             : test_f1,
    "history"             : history,
}, os.path.join(SAVE_DIR, "lstm_full.pt"))

# Weights only (lighter, for inference)
torch.save(model.state_dict(), os.path.join(SAVE_DIR, "lstm_weights_only.pt"))

# Config + label map as JSON
cfg_json = {k: v for k, v in CONFIG.items() if not callable(v)}
cfg_json["label_map"]  = LABEL_MAP
cfg_json["label_imap"] = {str(k): v for k, v in LABEL_IMAP.items()}
with open(os.path.join(SAVE_DIR, "config.json"), "w") as f:
    json.dump(cfg_json, f, indent=2)

for fname in ["lstm_full.pt", "lstm_weights_only.pt", "config.json"]:
    fpath = os.path.join(SAVE_DIR, fname)
    mb    = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {fname:<35}  {mb:.1f} MB")

print(f"\n  Artifacts saved to: {SAVE_DIR}")
print(SEP)


Loading best checkpoint: /content/drive/MyDrive/Colab Notebooks/MVSA_Single/lstm_checkpoints/lstm_best.pt
  Checkpoint epoch : 2   val_f1=0.5893   val_acc=0.6674


Testing:   0%|          | 0/15 [00:00<?, ?it/s]


  Test Set Results — Multimodal LSTM + Co-Attention
  Accuracy  (overall)    : 0.6969  (69.69%)
  Macro F1  (unweighted) : 0.6318

  Per-Class Classification Report:
              precision    recall  f1-score   support

    negative     0.5957    0.6176    0.6065       136
     neutral     0.4902    0.5319    0.5102        47
    positive     0.7923    0.7658    0.7788       269

    accuracy                         0.6969       452
   macro avg     0.6261    0.6385    0.6318       452
weighted avg     0.7018    0.6969    0.6990       452

  Confusion Matrix  (rows=true, cols=pred):
              negative     neutral    positive
    negative          84           9          43
     neutral          11          25          11
    positive          46          17         206

  Training History (last 5 epochs):
 epoch  train_loss  train_acc  val_loss  val_acc   val_f1
     2    0.710511   0.692350  0.913852 0.667406 0.589344
     3    0.490879   0.796009  1.037631 0.600887 0.541942
   